In [2]:
"""
COMPLETE ASSIGNMENT 2: Modern RNNs for Time Series Forecasting
CS511 - Advanced Artificial Intelligence

Author: Syed Muhammad Murtaza Shah
Student ID: 250302343

This script contains all tasks (A-E) in one file.
Run: python run_assignment2.py
"""

# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import time
import os
import random
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

LOOKBACK = 96          # L: Number of past steps to look back
HORIZON = 24           # H: Number of future steps to predict
BATCH_SIZE = 64
EPOCHS = 20
LR = 0.001
PATIENCE = 7           # Early stopping patience
SEED = 42

# Create results directory
os.makedirs("results", exist_ok=True)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# TASK A: DATA PIPELINE
# ============================================================

def set_seed(seed=SEED):
    """Fix random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class WindowedTimeSeries(Dataset):
    """Sliding window dataset for time series forecasting"""
    
    def __init__(self, features, targets, lookback=LOOKBACK, horizon=HORIZON):
        self.features = features
        self.targets = targets
        self.L = lookback
        self.H = horizon
    
    def __len__(self):
        return len(self.features) - self.L - self.H + 1
    
    def __getitem__(self, idx):
        x = self.features[idx:idx + self.L]           # (L, d)
        y = self.targets[idx + self.L:idx + self.L + self.H]  # (H, 1)
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))

def load_and_prepare_data():
    """Load ETTh1 dataset and prepare train/val/test splits"""
    
    print("\n" + "="*70)
    print("TASK A: Data Pipeline")
    print("="*70)
    
    # Load dataset
    url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
    print(f"\nLoading data from: {url}")
    df = pd.read_csv(url)
    
    # Parse date
    df['date'] = pd.to_datetime(df['date'])
    dates = df['date']
    data = df.drop(columns=['date'])
    
    feature_names = data.columns.tolist()
    print(f"Features: {feature_names}")
    print(f"Data shape: {data.shape}")
    print(f"Date range: {dates.min()} to {dates.max()}")
    
    # Check for missing values
    if data.isnull().sum().sum() > 0:
        print(f"Warning: {data.isnull().sum().sum()} missing values found. Filling with forward fill.")
        data = data.fillna(method='ffill').fillna(method='bfill')
    
    data_values = data.values
    n = len(data_values)
    
    # Split: 70% train, 10% val, 20% test
    train_end = int(n * 0.7)
    val_end = int(n * 0.8)
    
    train_data = data_values[:train_end]
    val_data = data_values[train_end:val_end]
    test_data = data_values[val_end:]
    
    print(f"\nSplit sizes:")
    print(f"  Train: {len(train_data)} samples ({len(train_data)/n*100:.1f}%)")
    print(f"  Val:   {len(val_data)} samples ({len(val_data)/n*100:.1f}%)")
    print(f"  Test:  {len(test_data)} samples ({len(test_data)/n*100:.1f}%)")
    
    # Normalize using training statistics only
    scaler = StandardScaler()
    train_data_norm = scaler.fit_transform(train_data)
    val_data_norm = scaler.transform(val_data)
    test_data_norm = scaler.transform(test_data)
    
    # Target: OT (oil temperature) is the last column
    target_idx = feature_names.index('OT')
    print(f"\nTarget variable: OT (index {target_idx})")
    
    train_target = train_data_norm[:, target_idx:target_idx+1]
    val_target = val_data_norm[:, target_idx:target_idx+1]
    test_target = test_data_norm[:, target_idx:target_idx+1]
    
    # Features: all variables
    train_features = train_data_norm
    val_features = val_data_norm
    test_features = test_data_norm
    
    # Create datasets
    train_dataset = WindowedTimeSeries(train_features, train_target)
    val_dataset = WindowedTimeSeries(val_features, val_target)
    test_dataset = WindowedTimeSeries(test_features, test_target)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"\nDataset sizes:")
    print(f"  Train samples: {len(train_dataset)}")
    print(f"  Val samples:   {len(val_dataset)}")
    print(f"  Test samples:  {len(test_dataset)}")
    
    return train_loader, val_loader, test_loader, scaler, data_values.shape[1], feature_names, target_idx

# ============================================================
# TASK B: BASELINE GRU MODEL
# ============================================================

class GRUForecast(nn.Module):
    """Baseline GRU forecaster for multi-step time series prediction"""
    
    def __init__(self, input_dim, hidden_dim=64, horizon=HORIZON, out_dim=1, num_layers=2, dropout=0.2):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.horizon = horizon
        self.out_dim = out_dim
        self.num_layers = num_layers
        
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, horizon * out_dim)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.head.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
    
    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_hidden = gru_out[:, -1, :]
        predictions = self.head(last_hidden)
        return predictions.view(-1, self.horizon, self.out_dim)
    
    def get_params_count(self):
        return sum(p.numel() for p in self.parameters())

# ============================================================
# TASK C2: iGRU VARIANT (Cross-Channel Mixing)
# ============================================================

class ChannelMixer(nn.Module):
    """Cross-channel mixing block to capture dependencies between variables"""
    
    def __init__(self, input_dim, hidden_dim=None):
        super().__init__()
        
        if hidden_dim is None:
            hidden_dim = input_dim
        
        self.mixer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, input_dim),
            nn.LayerNorm(input_dim)
        )
    
    def forward(self, x):
        B, L, d = x.shape
        x_flat = x.view(B * L, d)
        mixed = self.mixer(x_flat)
        mixed = mixed.view(B, L, d)
        return mixed + x  # Residual connection

class iGRUForecast(nn.Module):
    """iGRU: Improved GRU with cross-channel mixing"""
    
    def __init__(self, input_dim, hidden_dim=64, horizon=HORIZON, out_dim=1, num_layers=2, dropout=0.2):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.horizon = horizon
        self.out_dim = out_dim
        
        self.channel_mixer = ChannelMixer(input_dim)
        
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, horizon * out_dim)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.head.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
    
    def forward(self, x):
        x_mixed = self.channel_mixer(x)
        gru_out, _ = self.gru(x_mixed)
        last_hidden = gru_out[:, -1, :]
        predictions = self.head(last_hidden)
        return predictions.view(-1, self.horizon, self.out_dim)
    
    def get_params_count(self):
        return sum(p.numel() for p in self.parameters())

# ============================================================
# METRICS (RMSE, MAE, MAPE)
# ============================================================

def rmse(y_pred, y_true):
    return torch.sqrt(torch.mean((y_pred - y_true) ** 2)).item()

def mae(y_pred, y_true):
    return torch.mean(torch.abs(y_pred - y_true)).item()

def mape(y_pred, y_true, eps=1e-6):
    mask = torch.abs(y_true) > 0.1
    if mask.sum() == 0:
        return 0.0
    ape = torch.abs((y_pred - y_true)[mask] / (y_true[mask] + eps))
    return (ape.mean()).item() * 100

def calculate_metrics(y_pred, y_true):
    return {
        'RMSE': rmse(y_pred, y_true),
        'MAE': mae(y_pred, y_true),
        'MAPE': mape(y_pred, y_true)
    }

# ============================================================
# EARLY STOPPING
# ============================================================

class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            return False
        
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
            return False

# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_model(model, model_name, train_loader, val_loader):
    """Train a single model with early stopping"""
    
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    print(f"Parameters: {model.get_params_count():,}")
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    criterion = nn.MSELoss()
    early_stopping = EarlyStopping()
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    start_time = time.time()
    
    for epoch in range(EPOCHS):
        # Training
        model.train()
        train_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False):
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            y_pred = model(x)
            loss = criterion(y_pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                y_pred = model(x)
                val_loss += criterion(y_pred, y).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"results/best_{model_name}.pt")
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Early stopping
        if early_stopping(avg_val_loss):
            print(f"  Early stopping at epoch {epoch+1}")
            break
        
        print(f"\nEpoch {epoch+1}/{EPOCHS}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")
    
    training_time = time.time() - start_time
    
    # Load best model
    model.load_state_dict(torch.load(f"results/best_{model_name}.pt"))
    
    return model, train_losses, val_losses, training_time

# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, model_name, test_loader):
    """Evaluate model on test set"""
    
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Testing", leave=False):
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            all_preds.append(y_pred.cpu())
            all_targets.append(y.cpu())
    
    y_pred = torch.cat(all_preds, dim=0)
    y_true = torch.cat(all_targets, dim=0)
    
    metrics = calculate_metrics(y_pred, y_true)
    
    return metrics, y_pred, y_true

# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_training_curves(train_losses, val_losses, model_name):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title(f'Training Curves - {model_name}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f'results/loss_curve_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: results/loss_curve_{model_name}.png")

def plot_predictions(y_true, y_pred, model_name, num_samples=3):
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        axes[i].plot(y_true[i, :, 0].numpy(), label='Ground Truth', linewidth=2)
        axes[i].plot(y_pred[i, :, 0].numpy(), label='Prediction', linewidth=2, linestyle='--')
        axes[i].set_title(f'Sample {i+1} - {model_name}')
        axes[i].set_xlabel('Forecast Horizon (hours)')
        axes[i].set_ylabel('Normalized Value')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'results/predictions_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: results/predictions_{model_name}.png")

def plot_comparison_bar(results):
    models = list(results.keys())
    rmse_values = [results[m]['RMSE'] for m in models]
    mae_values = [results[m]['MAE'] for m in models]
    
    x = np.arange(len(models))
    width = 0.35
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.bar(x - width/2, rmse_values, width, label='RMSE', color='#FF6B6B')
    ax1.bar(x + width/2, mae_values, width, label='MAE', color='#4ECDC4')
    ax1.set_xlabel('Model')
    ax1.set_ylabel('Error')
    ax1.set_title('Error Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(models)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    mape_values = [results[m]['MAPE'] for m in models]
    ax2.bar(models, mape_values, color='#45B7D1')
    ax2.set_xlabel('Model')
    ax2.set_ylabel('MAPE (%)')
    ax2.set_title('MAPE Comparison')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/comparison_bar.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: results/comparison_bar.png")

# ============================================================
# EFFICIENCY PROFILING (TASK E)
# ============================================================

def efficiency_profile(model, model_name, input_shape=(BATCH_SIZE, LOOKBACK, 7)):
    """Profile model efficiency"""
    
    model = model.to(device)
    model.eval()
    
    # Parameter count
    params = model.get_params_count()
    
    # Inference time
    x = torch.randn(*input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(10):
            _ = model(x)
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    n_runs = 200
    
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(x)
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    inference_time = (time.time() - start) / n_runs * 1000  # ms
    
    # Training time per batch (estimate)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    y = torch.randn(BATCH_SIZE, HORIZON, 1).to(device)
    
    # Warm-up
    for _ in range(5):
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    n_batches = 50
    
    for _ in range(n_batches):
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    train_time = (time.time() - start) / n_batches * 1000  # ms
    
    return {
        'Parameters': params,
        'Train Time/Batch (ms)': train_time,
        'Inference Time/Batch (ms)': inference_time
    }

# ============================================================
# MAIN FUNCTION
# ============================================================

def main():
    """Run complete assignment"""
    
    print("\n" + "="*70)
    print("CS511 ASSIGNMENT 2: Modern RNNs for Time Series Forecasting")
    print("="*70)
    print(f"Configuration:")
    print(f"  Lookback (L): {LOOKBACK}")
    print(f"  Horizon (H): {HORIZON}")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Max Epochs: {EPOCHS}")
    print(f"  Learning Rate: {LR}")
    print(f"  Early Stopping Patience: {PATIENCE}")
    print(f"  Random Seed: {SEED}")
    print("="*70)
    
    # Set seed for reproducibility
    set_seed()
    
    # Load data
    train_loader, val_loader, test_loader, scaler, input_dim, feature_names, target_idx = load_and_prepare_data()
    
    # Define models
    models = {
        'GRU': GRUForecast(input_dim=input_dim),
        'iGRU': iGRUForecast(input_dim=input_dim)
    }
    
    results = {}
    
    # Train and evaluate each model
    for name, model in models.items():
        print(f"\n{'='*70}")
        print(f"Processing {name}")
        print(f"{'='*70}")
        
        # Train
        trained_model, train_losses, val_losses, train_time = train_model(
            model, name, train_loader, val_loader
        )
        
        # Plot training curves
        plot_training_curves(train_losses, val_losses, name)
        
        # Evaluate
        metrics, y_pred, y_true = evaluate_model(trained_model, name, test_loader)
        metrics['Training Time (s)'] = train_time
        metrics['Parameters'] = trained_model.get_params_count()
        
        # Plot predictions
        plot_predictions(y_true, y_pred, name)
        
        # Efficiency profile
        efficiency = efficiency_profile(trained_model, name)
        metrics.update(efficiency)
        
        results[name] = metrics
        
        print(f"\n{name} Results:")
        for key, value in metrics.items():
            print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
    
    # Comparison plot
    plot_comparison_bar(results)
    
    # Final comparison table
    print("\n" + "="*70)
    print("FINAL COMPARISON TABLE")
    print("="*70)
    print(f"{'Model':<10} {'RMSE':<12} {'MAE':<12} {'MAPE(%)':<12} {'Params':<12} {'Train(s)':<12} {'Inf(ms)':<12}")
    print("-" * 85)
    
    for name, metrics in results.items():
        print(f"{name:<10} {metrics['RMSE']:<12.4f} {metrics['MAE']:<12.4f} "
              f"{metrics['MAPE']:<12.2f} {metrics['Parameters']:<12,} "
              f"{metrics['Training Time (s)']:<12.1f} {metrics['Inference Time/Batch (ms)']:<12.2f}")
    
    # Save results to CSV
    df = pd.DataFrame(results).T
    df.to_csv('results/final_results.csv')
    print("\n✅ Results saved to results/final_results.csv")
    
    print("\n" + "="*70)
    print("ALL TASKS COMPLETED SUCCESSFULLY!")
    print("="*70)
    print("\nGenerated files in 'results/' directory:")
    print("  - loss_curve_GRU.png")
    print("  - loss_curve_iGRU.png")
    print("  - predictions_GRU.png")
    print("  - predictions_iGRU.png")
    print("  - comparison_bar.png")
    print("  - final_results.csv")
    print("  - best_GRU.pt")
    print("  - best_iGRU.pt")
    
    return results

if __name__ == "__main__":
    results = main()

Using device: cpu

CS511 ASSIGNMENT 2: Modern RNNs for Time Series Forecasting
Configuration:
  Lookback (L): 96
  Horizon (H): 24
  Batch Size: 64
  Max Epochs: 20
  Learning Rate: 0.001
  Early Stopping Patience: 7
  Random Seed: 42

TASK A: Data Pipeline

Loading data from: https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv
Features: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
Data shape: (17420, 7)
Date range: 2016-07-01 00:00:00 to 2018-06-26 19:00:00

Split sizes:
  Train: 12194 samples (70.0%)
  Val:   1742 samples (10.0%)
  Test:  3484 samples (20.0%)

Target variable: OT (index 6)

Dataset sizes:
  Train samples: 12075
  Val samples:   1623
  Test samples:  3365

Processing GRU

Training GRU
Parameters: 41,848



Epoch 1/20: Train Loss=0.3146, Val Loss=0.1409



Epoch 2/20: Train Loss=0.1574, Val Loss=0.1579



Epoch 3/20: Train Loss=0.1398, Val Loss=0.1613



Epoch 4/20: Train Loss=0.1307, Val Loss=0.1906


...


Epoch 10/20: Train Loss=0.1051, Val Loss=0.2658



Epoch 11/20: Train Loss=0.1001, Val Loss=0.1307



Epoch 12/20: Train Loss=0.0974, Val Loss=0.2003


  Early stopping at epoch 13
  Saved: results/loss_curve_iGRU.png


  Saved: results/predictions_iGRU.png

iGRU Results:
  RMSE: 0.2252
  MAE: 0.1635
  MAPE: 18.2092
  Training Time (s): 1331.1152
  Parameters: 41974
  Train Time/Batch (ms): 423.2815
  Inference Time/Batch (ms): 104.3651
  Saved: results/comparison_bar.png

FINAL COMPARISON TABLE
Model      RMSE         MAE          MAPE(%)      Params       Train(s)     Inf(ms)     
-------------------------------------------------------------------------------------
GRU        0.2221       0.1670       19.72        41,848       1345.1       128.32      
iGRU       0.2252       0.1635       18.21        41,974       1331.1       104.37      

✅ Results saved to results/final_results.csv

ALL TASKS COMPLETED SUCCESSFULLY!

Generated files in 'results/' directory:
  - loss_curve_GRU.png
  - loss_curve_iGRU.png
  - predictions_GRU.png
  - predictions_iGRU.png
  - comparison_bar.png
  - final_results.csv
  - best_GRU.pt
  - best_iGRU.pt
